# Notebook 1 — Phase 1: arXiv Fetching, Docling GPU Parsing & Qdrant Cloud Ingestion

**Kaggle Settings required (right sidebar):**
- Accelerator: **GPU T4 x2** (or **GPU P100**)
- Internet: **On**
- Add-ons → Secrets: tick **`QDRANT_URL`** and **`QDRANT_API_KEY`**

*Tip: Use **Save & Run All (Commit)** so Kaggle runs in the background without idle timeout.*

In [ ]:
# Cell 1: Clone or pull the latest repo from GitHub
from pathlib import Path

REPO_URL = "https://github.com/Nody3112V/AI-papers-agentic-RAG.git"
REPO_DIR = Path("/kaggle/working/AI-papers-agentic-RAG")

if REPO_DIR.exists():
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

In [ ]:
# Cell 2: Install Kaggle GPU ingestion dependencies
!pip install -q arxiv docling pymupdf4llm qdrant-client fastembed-gpu pyyaml pydantic-settings

In [ ]:
# Cell 3: Load Qdrant Cloud secrets securely from Kaggle Secrets
import os

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["QDRANT_MODE"] = "cloud"
os.environ["QDRANT_URL"] = secrets.get_secret("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = secrets.get_secret("QDRANT_API_KEY")
print("Secrets loaded successfully (QDRANT_MODE=cloud).")

In [ ]:
# Cell 4: Initialize papers.db and fetch 60 AI papers from arXiv (4 topic clusters x 15)
!python scripts/init_db.py --db-path data/sqlite/papers.db
!python scripts/fetch_arxiv_papers.py --limit 60 --db-path data/sqlite/papers.db --pdf-dir /kaggle/working/raw_papers

In [ ]:
# Cell 5: Run Docling GPU parsing + table-aware chunking + idempotent upsert to Qdrant Cloud
!python scripts/run_ingestion.py \
    --device cuda \
    --target cloud \
    --parser docling \
    --db-path data/sqlite/papers.db \
    --markdown-dir data/processed/markdown \
    --export-local-backup

In [ ]:
# Cell 6: Verify 100% idempotency (re-running upsert must keep points_count identical)
from src.ingestion.indexer import get_qdrant_client

client = get_qdrant_client(mode="cloud")
before = client.get_collection("ai_papers_chunks").points_count
!python scripts/run_ingestion.py --device cuda --target cloud --limit 5 --db-path data/sqlite/papers.db --markdown-dir data/processed/markdown
after = client.get_collection("ai_papers_chunks").points_count
print(f"Idempotency check: before={before} | after={after} | equal={before == after}")
assert before == after, "Point count changed on re-upsert!"

In [ ]:
# Cell 7: Package lightweight output (~10-12 MB) to download back to Laptop
import shutil
from pathlib import Path

export_dir = Path("/kaggle/working/phase1_export")
if export_dir.exists():
    shutil.rmtree(export_dir)

(export_dir / "sqlite").mkdir(parents=True, exist_ok=True)
shutil.copy2("data/sqlite/papers.db", export_dir / "sqlite" / "papers.db")
shutil.copytree("data/processed/markdown", export_dir / "processed" / "markdown")

archive_path = shutil.make_archive("/kaggle/working/phase1_data_for_laptop", "zip", export_dir)
size_mb = Path(archive_path).stat().st_size / (1024 * 1024)
print(f"Ready to download: {archive_path} ({size_mb:.2f} MB)")